## Tutorial on full minimal coupling within the self-consistent Maxwell-TDDFT framework

### Authors: Franco Bonafé, Esra Ilke Albar, Carlos Bustamante

In order to account for light-matter interactions beyond the electric dipole approximation, either with an external field or with the self-induced electromagnetic fields, we need to couple real-time TDDFT with the Maxwell solver in Octopus. This can be done by using the multisystem features, and defining an electronic system plus a Maxwell system. However, if the goal is to study the beyond-dipole effects with an externally-prescribed field that is known to be a solution of Maxwell's equations (such as plane waves or Bessel beams), then an analytical external source can be prescribed, without propagating these incident waves in the Maxwell box. This could have several benefits, as it will be seen below.

In this tutorial we will run:

1. the **ground state** of a benzene molecule (`GS_DIR`),
2. a time propagation with the external field coupled at the **dipole** level (`EXT_SOURCE_DIPOLE_DIR`),
3. a time propagation with a **plane wave in full minimal coupling** (`PLANE_WAVE_FULL_MINIMAL_DIR`),
4. a time propagation with a **Bessel beam in full minimal coupling** (`BESSEL_FULL_MINIMAL_DIR`).

All of them use the analytical external source, i.e. no Maxwell box is propagated.

### 0. Light-matter coupling Hamiltonians

The theory behind this tutorial is described in detail in

> F. P. Bonafé, E. I. Albar, S. T. Ohlmann, V. P. Kosheleva, C. M. Bustamante, F. Troisi, A. Rubio and H. Appel, *Full minimal coupling Maxwell-TDDFT: an ab initio framework for light-matter interaction beyond the dipole approximation*, **Phys. Rev. B 111, 085114 (2025)**.

The starting point is the time-dependent Kohn-Sham equation written with the **full minimal coupling (f.m.c.)** Hamiltonian in velocity gauge (SI units, as in the paper):

$$
\mathcal{H}^{\mathrm{f.m.c.}} = \frac{1}{2m}\left(-i\hbar\nabla + \frac{|e|}{c}\mathbf{A}(\mathbf{r},t)\right)^{2}
+ \frac{|e|}{m}\mathbf{B}(\mathbf{r},t)\cdot\hat{\mathbf{s}}
+ V_{\mathrm{H}}[n](\mathbf{r},t) + V_{\mathrm{xc}}[n](\mathbf{r},t) + V_{\mathrm{nuc}}(\mathbf{r},t)
$$

Here $\mathbf{A}(\mathbf{r},t)$ is the **transverse** vector potential, which retains its full space **and** time dependence, and $\mathbf{B}=\nabla\times\mathbf{A}$. It contains both the externally prescribed field and the field induced by matter,
$\mathbf{A}(\mathbf{r},t) = \mathbf{A}_{\mathrm{ext}}(\mathbf{r},t) + \mathbf{A}_{\mathrm{mat}}(\mathbf{r},t)$.
Because no Taylor expansion of $\mathbf{A}$ is performed, this Hamiltonian contains **all multipolar orders at once** and the resulting observables are origin-independent.

#### The dipole Hamiltonians

The paper compares the f.m.c. Hamiltonian against the two "dipole-level" Hamiltonians that are normally used in electronic-structure codes. Both are obtained by expanding the fields around a chosen center $\mathbf{r}_0$ (typically the center of mass) and truncating.

**(a) Electric dipole in velocity gauge.** The vector potential is replaced by its spatial average over the matter box, $\mathbf{A}(\mathbf{r},t)\rightarrow\mathbf{A}(t)$, so that the interaction reduces to

$$
H_{\mathrm{int}}^{\mathrm{dip}} = \frac{1}{2m}\left(2\,\mathbf{A}(t)\cdot\hat{\mathbf{p}} + \mathbf{A}^{2}(t)\right)
$$

All spatial structure of the field is lost: every point of the molecule feels the same field at the same time, so there is no magnetic dipole, no electric quadrupole, and no retardation across the molecule. This is `MaxwellCouplingMode = velocity_gauge_dipole`.

**(b) Truncated multipolar expansion in length gauge.** Expanding $\mathbf{A}$ and $\mathbf{B}$ around $\mathbf{r}_0$ and keeping up to second order gives

$$
\mathcal{H}^{\mathrm{m.e.}} = -\frac{\hbar^{2}}{2m}\nabla^{2}
+ |e|\,\mathbf{r}\cdot\mathbf{E}_{\perp}(\mathbf{r}_{0},t)
+ i\frac{|e|}{2m}\mathbf{B}(\mathbf{r}_{0},t)\cdot(\mathbf{r}\times\nabla)
+ \left.\frac{1}{2}|e|(\mathbf{r}\cdot\nabla)\,\mathbf{r}\cdot\mathbf{E}_{\perp}(\mathbf{r},t)\right|_{\mathbf{r}=\mathbf{r}_{0}}
+ V_{\mathrm{H}} + V_{\mathrm{xc}} + V_{\mathrm{nuc}}
$$

where the second term is the familiar **electric dipole** coupling, the third one is the **magnetic dipole** coupling,

$$
i\frac{|e|}{2m}\mathbf{B}(\mathbf{r}_{0},t)\cdot(\mathbf{r}\times\nabla) = -\frac{|e|}{2m}\mathbf{B}(\mathbf{r}_{0},t)\cdot\hat{\mathbf{L}}
$$

and the fourth one is the **electric quadrupole** coupling,
$\frac{1}{2}|e|\sum_{ij} r_i\,\mathbb{Q}_{ij}\,r_j$, with $\mathbb{Q}_{ij}=\partial_i \mathbf{E}_{\perp,j}(\mathbf{r},t)|_{\mathbf{r}=\mathbf{r}_0}$ the electric field gradient tensor. This is `MaxwellCouplingMode = multipolar_expansion`, and the terms to be included are selected with `MultipolarExpansionTerms`.

Truncating at any finite multipole order makes the observables **origin-dependent**, and the convergence with the multipole order can be slow for strongly inhomogeneous fields. One of the results of the paper is that, for benzene driven at $\omega = 270$ eV, the full minimal coupling dipole moment is identical for a centered and a displaced molecule, while the truncated multipolar treatment is not.

Finally, the back-reaction (not used in this tutorial, where only forward coupling is considered) is generated by the **full** current density, which within f.m.c. contains a paramagnetic, a diamagnetic and a magnetization term:

$$
\mathbf{J}(\mathbf{r},t) = \frac{e\hbar}{2im}\sum_{j\sigma}\left(\left[\nabla\varphi_j^{\dagger}\right]\varphi_j - \varphi_j^{\dagger}\left[\nabla\varphi_j\right]\right)
- \frac{e^{2}}{mc}\mathbf{A}(\mathbf{r},t)\,n(\mathbf{r},t)
- \frac{e\hbar}{2m}\nabla\times\sum_{j\sigma}\varphi_j^{\dagger}\hat{\mathbf{s}}\varphi_j
$$

| `MaxwellCouplingMode` | field used | orders included | origin-independent |
|---|---|---|---|
| `velocity_gauge_dipole` | $\mathbf{A}(t)$ | electric dipole | yes (trivially) |
| `multipolar_expansion` | $\mathbf{E}_\perp(\mathbf{r}_0,t)$, $\mathbf{B}(\mathbf{r}_0,t)$, $\mathbb{Q}$ | up to 2nd order | no |
| `full_minimal_coupling` | $\mathbf{A}(\mathbf{r},t)$ | all | yes |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read
from postopus import Run

# ----------------------------------------------------------------------
# Octopus executable and number of MPI processes.
# Change these two lines to match your installation; they are exported to
# the environment so that the %%bash cells below can use $OCTOPUS / $NPROC.
# ----------------------------------------------------------------------
OCTOPUS = "/home/spectrodyn_user/Codes/octopus/_build/app/bin/octopus"
NPROC   = 6

os.environ["OCTOPUS"] = OCTOPUS
os.environ["NPROC"]   = str(NPROC)

print("octopus executable :", OCTOPUS, "(found)" if os.path.exists(OCTOPUS) else "(NOT FOUND!)")

### 0.1 The external field: temporal vs. spatio-temporal profile

Before running anything, it is useful to look at the field that we are going to prescribe. In the `ExternalSource` blocks below we define a **vector potential** plane wave with a Gaussian envelope. Both the carrier and the envelope depend on space and time only through the retarded coordinate $u = \hat{\mathbf{k}}\cdot\mathbf{r} - c\,t ,$ which is what makes the expression an exact solution of the free Maxwell equations:

$$
\mathbf{A}(\mathbf{r},t) = A_0\,\hat{\boldsymbol{\varepsilon}}\;
\cos\!\left(|\mathbf{k}|\,u + \phi\right)\;
\exp\!\left[-\frac{(u-s_0)^{2}}{2\sigma^{2}}\right],
\qquad
\mathbf{E}(\mathbf{r},t) = -\frac{1}{c}\frac{\partial \mathbf{A}}{\partial t} = \frac{\partial \mathbf{A}}{\partial u}
$$

with $A_0 = -A_{\mathrm{ampl}}c/\omega$, $\hat{\boldsymbol{\varepsilon}}=\hat{x}$ (polarization), $\hat{\mathbf{k}}=\hat{y}$ (propagation), $|\mathbf{k}|=\omega/c$, $\phi = -\pi/2$, $s_0 = -c\,t_0$ (`p_s`) and $\sigma = c\,\tau_0$ (`pw`). These are exactly the numbers that appear in the `MaxwellIncidentWaves` and `MaxwellFunctions` blocks of the input files.

Note that the field is polarized along $x$ and travels along $y$, so below we plot $E_x$ **as a function of the propagation coordinate**. All quantities are in atomic units.

In [ ]:
# ----------------------------- atomic units -----------------------------
c_au   = 137.035999            # speed of light in a.u.
eV     = 1.0 / 27.211386245988 # 1 eV in Hartree
fs     = 41.341373335182114    # 1 fs in a.u. of time

# ------------------- same parameters as used later for the tutorial---------------
omega     = 270 * eV
period    = 0.015 * fs
ampl      = 0.0001
tau0      = 1.0 * period       # sigma of the Gaussian envelope (in time)
t0        = 3 * tau0           # time at which the pulse peak reaches r = 0
p_s       = -t0 * c_au         # spatial center of the envelope at t = 0
pw        = tau0 * c_au        # spatial width of the envelope
phase_vec = -np.pi / 2

k_hat = np.array([0.0, 1.0, 0.0])   # propagation direction
pol   = np.array([1.0, 0.0, 0.0])   # polarization direction
k_abs = omega / c_au
A0    = -ampl * c_au / omega

def retarded_coord(r, t):
    """u = k_hat . r - c t  (r may be a single point or an (N,3) array)."""
    return np.tensordot(np.asarray(r, dtype=float), k_hat, axes=([-1], [0])) - c_au * t

def vector_potential(r, t):
    """Vector potential of the pulse, returns the amplitude along `pol`."""
    u   = retarded_coord(r, t)
    env = np.exp(-((u - p_s) ** 2) / (2.0 * pw ** 2))
    return A0 * env * np.cos(k_abs * u + phase_vec)

def electric_field(r, t):
    """E = -(1/c) dA/dt = dA/du ; returns the amplitude along `pol`."""
    u    = retarded_coord(r, t)
    env  = np.exp(-((u - p_s) ** 2) / (2.0 * pw ** 2))
    car  = np.cos(k_abs * u + phase_vec)
    dcar = -k_abs * np.sin(k_abs * u + phase_vec)
    denv = -(u - p_s) / pw ** 2 * env
    return A0 * (denv * car + env * dcar)

print(f"omega      = {omega:.4f} Ha  ({omega/eV:.1f} eV)")
print(f"period     = {period:.4f} a.u.   2*pi/omega = {2*np.pi/omega:.4f} a.u.")
print(f"|k|        = {k_abs:.5f} a.u.^-1   ->  lambda = {2*np.pi/k_abs:.1f} bohr")
print(f"A0         = {A0:.3e} a.u.   (peak |E| = {ampl:.1e} a.u.)")
print(f"envelope   : center s0 = {p_s:.1f} bohr, width sigma = {pw:.1f} bohr")

In [ ]:
# ---------- E_x along the propagation axis, for several times ----------
s = np.linspace(-450, 450, 3000)         # coordinate along k_hat (here: y)
r = np.zeros((s.size, 3))
r[:, 1] = s                              # put the line along the y axis

times = [0.5 * t0, t0, 1.5 * t0, 2.0 * t0]

fig, ax = plt.subplots(figsize=(9, 4.5))
for t in times:
    ax.plot(s, electric_field(r, t) * 1e4,
            label=rf"$t$ = {t:.2f} a.u. = {t/period:.1f} $T$")

# the electronic box is tiny compared with the pulse
Lbox = 3.5 / 0.529177
ax.axvspan(-Lbox, Lbox, color='0.85', zorder=0, label='electronic box')

ax.set_xlabel('propagation coordinate [bohr]')
ax.set_ylabel(r'$E_x \times 10^{4}$ [a.u.]')
ax.set_title('Plane-wave pulse travelling towards the molecule')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------- E_x as a function of time at the origin ----------------
t = np.linspace(0, 2 * t0, 4000)
E_origin = electric_field(np.zeros(3), t)
A_origin = vector_potential(np.zeros(3), t)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(t, E_origin * 1e4, label=r'$E_x \times 10^{4}$')
ax.plot(t, A_origin * 1e4 * k_abs, '--', lw=1,
        label=r'$|k|\,A_x \times 10^{4}$')
ax.axvline(t0, color='k', lw=0.8, ls=':')
ax.text(t0, ax.get_ylim()[1] * 0.9, r'  $t_0$', fontsize=10)

ax.set_xlabel('time [a.u.]')
ax.set_ylabel('field [a.u.]')
ax.set_title(r'Waveform seen at $\mathbf{r}=0$ (center of the molecule)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

Because the wavelength ($\approx 87$ bohr at 270 eV) is only about one order of magnitude larger than the molecule ($\approx 13$ bohr across), the field is **not** constant over the molecule. That is precisely the regime where the dipole approximation starts to fail, and it is what we test in the next sections.

### 1. XUV light-driven dynamics in benzene

As beyond-dipole effects are more evident for shorter wavelengths, we focus on this tutorial on the XUV response of benzene. We will simulate the incidence of an external laser on a benzene molecule oriented in the $xy$ plane. For this tutorial we will have to consider the electronic structure of the molecule with all-electron potentials, instead of using pseudopotentials, as applying a space-dependent vector potential implies a gauge transformation that has problems (a path ambiguity) when using non-local potentials, as it is the case for pseudopotentials (see (1), (2)). For this we sill use the "full delta" description of the potential. 

We need to run first the ground state calculation uisng the following input file. For more details on the basic features of Octopus, please refer to the basic tutorials (https://www.octopus-code.org/documentation/14/tutorial/basics/).

In [ ]:
if not os.path.exists('GS_DIR'):
    os.mkdir('GS_DIR')

In [ ]:
%%writefile GS_DIR/benzene.xyz
12
units: A
      C                    0.000000    1.364239    0.000000
      C                    1.191347    0.685530    0.000000
      C                    1.191347   -0.685530    0.000000
      C                    0.000000   -1.364239    0.000000
      C                   -1.191347   -0.685530    0.000000
      C                   -1.191347    0.685530    0.000000
      H                    0.000000    2.441134    0.000000
      H                    2.124739    1.238817    0.000000
      H                    2.124739   -1.238817    0.000000
      H                    0.000000   -2.441134    0.000000
      H                   -2.124739   -1.238817    0.000000
      H                   -2.124739    1.238817    0.000000

In [ ]:
%%writefile GS_DIR/inp

CalculationMode = gs
ExperimentalFeatures = yes
FromScratch = yes
Dimensions = 3

XYZCoordinates = "benzene.xyz"
UnitsXYZFiles = angstrom_units
AllElectronType = full_delta

ExtraStates = 1

BoxShape = parallelepiped
Spacing = 0.1*angstrom
%Lsize
 3.5*angstrom | 3.5*angstrom | 1.5*angstrom
%

In [ ]:
%%bash
cd GS_DIR
export OMP_NUM_THREADS=1
mpirun -np $NPROC $OCTOPUS >& out.log
tail -n 3 out.log

#### 1.1. Dipole-level coupling

We will start by calculating the system's response to the external field coupled at the dipole level, as a reference calculation. We will create a directory (`EXT_SOURCE_DIPOLE_DIR`) and create there the following input file:

In [ ]:
if not os.path.exists('EXT_SOURCE_DIPOLE_DIR'):
    os.mkdir('EXT_SOURCE_DIPOLE_DIR')

In [ ]:
%%writefile EXT_SOURCE_DIPOLE_DIR/inp

CalculationMode = td
ExperimentalFeatures = yes
FromScratch = yes
RestartWallTimePeriod = 10.01
ParStates = no

%Systems
 "benzene" | electronic
%

XYZCoordinates = "benzene.xyz"
UnitsXYZFiles = angstrom_units
AllElectronType = full_delta

BoxShape = parallelepiped
benzene.Spacing = 0.1*angstrom
%benzene.Lsize
 3.5*angstrom | 3.5*angstrom | 1.5*angstrom
%

%TDOutput
 maxwell_field
 multipoles
%
%Output
 density | plane_z
%
OutputInterval = 20

omega = 270*ev
period = 0.015*fs
ampl = 0.0001
tau0 = 1*period # sigma
t0 = tau0*3
p_s = - t0*c
pw  = tau0*c
phase_vec = -pi/2

TDSystemPropagator = prop_aetrs
benzene.TDTimeStep = 0.008
TDPropagationTime = 7*period

MaxwellCouplingMode = velocity_gauge_dipole
AnalyticalExternalSource = yes
%ExternalSource.MaxwellIncidentWaves
 plane_wave_mx_function | vector_potential | -ampl*c/omega | 0 | 0 | "plane_waves_function" | phase_vec
%
%MaxwellFunctions
  "plane_waves_function" | mxf_gaussian_wave | 0 | omega/c | 0 | 0 | p_s | 0 | pw
%

In [ ]:
%%bash
mkdir -p EXT_SOURCE_DIPOLE_DIR/restart/benzene
cp -r GS_DIR/restart/* EXT_SOURCE_DIPOLE_DIR/restart/benzene
cd EXT_SOURCE_DIPOLE_DIR
cp ../GS_DIR/benzene.xyz .
export OMP_NUM_THREADS=1
mpirun -np $NPROC $OCTOPUS >& out.log
tail -n 3 out.log

It is important to note a few differences of this input files, with respect to the non-multisystem (also known as "legacy mode", covered in most of the basic Octopus tutorials) input files:

* `ExperimentalFeatures = yes` must be set to use the multisystem features
* The `Systems` block has to be properly defined, accounting for all the systems present in the calculation (in this case, we are going to couple to an external plane wave prescribed by a formula, and not propagated through the Maxwell solver, therefore we only have one system)
* The `TDSystemPropagator` option must be **mandatorily** set, otherwise the code will interpret that no propagation (also known as "static" propagator) wants to be used. Please note that this is a different option than the `TDPropagator` variable used in legacy-mode runs. Each system supports different propagators implemented in the new framework, e.g. the electronic system supports `prop_aetrs` and `prop_expmid`.
* The input variables can be introduced prepending the name of the system (defining the namespace). In this case, since we have only one system (except for the `ExternalSource` one, that is not a general system), it is not necessary to do it, it's only optional (as it is shown by defining `benzene.TDTimeStep`). When we have more than one system, it is mandatory to distinguish the input options with the namespace, otherwise they will be considered to be **global** and applied to all systems.
* The multisystem framework does not accept `TDMaxSteps` as a way to define the total simulation time (as different systems could have different time steps). Therefore we have to set `TDPropagationTime` instead.

In addition to the stardard multisystem options, we define here an `ExternalSource` block, that implements the formulae for external waves in real space and real time. The relevant input variables for this are:

* `AnalyticalExternalSource`: must be set to yes for the code to read the blocks prepended with `ExternalSource`.
* `ExternalSource.MaxwellIncidentWaves` allows to define an external wave using the same input block that is used to set the boundary conditions for a Maxwell run, but in this case the defined wave will be evaluated in the whole box, at every timestep. Here we are defining a vector potential plane wave with amplitude `ampl*c/omega` polarized in the _x_ direction, defined by an envelope function called `plane_waves_function` and having a phase `phase_vec`.
* `MaxwellFunctions`: this block allows to specify the spatio-temporal envelope function, and has been explained in the Maxwell tutorials. In this case it is a Guassian envelope which yields the following expression for the vector potential as a function of space and time:
$ A(\vec{r}, t) = A_0 \exp (i(\vec{k}\cdot\vec{r} - \omega t + \phi)) \exp(\frac{−(\vec{k}\cdot (\vec{r} - \vec{r}_0)/|\vec{k}|)^2}{2 \sigma^2})$
* `MaxwellCouplingMode` allows to define the coupling Hamiltonian with the external field. In this case, it defines the coupling to be at dipole level in velocity gauge, using the (transverse) dipole vector potential $A(t)$, obtained from the spatial average of $A(\vec{r},t)$:
$H_{int} = \frac{1}{2m} (2 \vec{A}(t).\hat{p} + \vec{A}^2(t))$

More information and other spatial envelope shapes can be found at the variable descriptions of `MaxwellIncidentWaves` and [MaxwellFunctions](https://www.octopus-code.org/documentation/14/variables/time-dependent/maxwellfunctions/).

**Important**: in order to use the external source, one has to define the external waves including the space-dependent part, both in the carrier and in the envelope. To learn how to convert from purely time-dependent fields specified via `TDExternalFields` to external source, please follow [this tutorial](./tutorial_ext_source.ipynb).


# 1.2. Plane wave full minimal coupling

Let's now repeat exactly the same calculation, with exactly the same plane wave, but coupling it to the electrons in **full minimal coupling** instead of the dipole approximation. We create the input file in the folder `PLANE_WAVE_FULL_MINIMAL_DIR`.

In this case, the `MaxwellCouplingMode` specifies full minimal coupling, which can be described by the following interaction Hamiltonian (compare with the dipole one in Section 0, where $\vec{A}$ does not depend on $\vec{r}$):

$H_{int} = \frac{1}{2m} (2 \vec{A}(\vec{r},t).\hat{p} + \vec{A}^2(\vec{r},t))$

In [ ]:
if not os.path.exists('PLANE_WAVE_FULL_MINIMAL_DIR'):
    os.mkdir('PLANE_WAVE_FULL_MINIMAL_DIR')

In [ ]:
%%writefile PLANE_WAVE_FULL_MINIMAL_DIR/inp

CalculationMode = td
ExperimentalFeatures = yes
FromScratch = yes
RestartWallTimePeriod = 10.01
ParStates = no

%Systems
 "benzene" | electronic
%

XYZCoordinates = "benzene.xyz"
UnitsXYZFiles = angstrom_units
AllElectronType = full_delta

BoxShape = parallelepiped
benzene.Spacing = 0.1*angstrom
%benzene.Lsize
 3.5*angstrom | 3.5*angstrom | 1.5*angstrom
%

%TDOutput
 multipoles
%
%Output
 density | plane_z
%
OutputInterval = 20

omega = 270*ev
period = 0.015*fs
ampl = 0.0001
tau0 = 1*period # sigma
t0 = tau0*3
p_s = - t0*c
pw  = tau0*c
phase_vec = -pi/2

TDSystemPropagator = prop_aetrs
benzene.TDTimeStep = 0.008
TDPropagationTime = 7*period

MaxwellCouplingMode = full_minimal_coupling
AnalyticalExternalSource = yes
%ExternalSource.MaxwellIncidentWaves
 plane_wave_mx_function | vector_potential | -ampl*c/omega | 0 | 0 | "plane_waves_function" | phase_vec
%
%MaxwellFunctions
  "plane_waves_function" | mxf_gaussian_wave | 0 | omega/c | 0 | 0 | p_s | 0 | pw
%

In [ ]:
%%bash
mkdir -p PLANE_WAVE_FULL_MINIMAL_DIR/restart/benzene
cp -r GS_DIR/restart/* PLANE_WAVE_FULL_MINIMAL_DIR/restart/benzene
cd PLANE_WAVE_FULL_MINIMAL_DIR
cp ../GS_DIR/benzene.xyz .
export OMP_NUM_THREADS=1
mpirun -np $NPROC $OCTOPUS >& out.log
tail -n 3 out.log

Let's see the differences between the two different coupling levels, by comparing the induced dipole moments for both runs with `ExternalSource`. We can use or adapt the following script:

In [ ]:
run_PLANE_WAVE_DIR = Run("PLANE_WAVE_FULL_MINIMAL_DIR") # Use Postopus to load all data produced by octopus
run_PLANE_WAVE_multipoles = run_PLANE_WAVE_DIR.benzene.td.multipoles() # Load multipole data

run_DIPOLE_DIR = Run("EXT_SOURCE_DIPOLE_DIR") # Use Postopus to load all data produced by octopus
run_DIPOLE_DIR_multipoles = run_DIPOLE_DIR.benzene.td.multipoles() # Load multipole data

diff_coupling = run_DIPOLE_DIR_multipoles["<x>(1)"] - run_PLANE_WAVE_multipoles["<x>(1)"] #  Calculate the difference between the two couplings
run_PLANE_WAVE_multipoles["diff_coupling"] = diff_coupling # Add the difference to one of the existing multipole data

In [ ]:
ax = run_PLANE_WAVE_multipoles.plot(x="t", y="<x>(1)", label="Plane wave, full minimal")
run_DIPOLE_DIR_multipoles.plot(x="t", y="<x>(1)", label='Plane wave, dipole', ax=ax);
run_PLANE_WAVE_multipoles.plot(x="t", y="diff_coupling", ax=ax, label = "(Full minimal - dipole) difference");
ax.set_ylabel('dipole moment <x> [a.u.]'); ax.set_xlabel('time [a.u.]');

In [ ]:
# Load density data with Postopus
run_DIPOLE_density = run_DIPOLE_DIR.benzene.td.density("z=0") # Load all density data
run_PLANE_WAVE_density = run_PLANE_WAVE_DIR.benzene.td.density("z=0") # Load all density data
#Calculate the density difference
diff_DIPOLE_density = run_DIPOLE_density - run_DIPOLE_density[0]
diff_PLANE_WAVE_density = run_PLANE_WAVE_density - run_PLANE_WAVE_density[0]
diff_coupling_density =  diff_PLANE_WAVE_density[:] - diff_DIPOLE_density

Define plot function for atom positions

In [ ]:
def plotxy_struc(ax, pos_x, pos_y):
    return ax.scatter(pos_x, pos_y, s=100, c='k', alpha=0.5, edgecolor='k')

# read atom positions
pos = read("GS_DIR/benzene.xyz").get_positions()/0.529177

Plot the density for each case

In [ ]:
cmap = plt.cm.terrain
vmax = 0.000001

idxs = (10,11,12)
fig, ax = plt.subplots(len(idxs),3,figsize=(15,12))

for ii,idx in enumerate(idxs):

    plots = [(diff_DIPOLE_density[idx], "Dipole"),
             (diff_PLANE_WAVE_density[idx], "Full minimal"),
             (diff_coupling_density[idx], "(Full minimal - dipole) difference")]

    for i, (data, title) in enumerate(plots):
        data.plot(cmap=cmap, ax=ax[ii,i], x="x", vmax=vmax, vmin=-vmax)
        ax[ii,i].set_title(title)
        plotxy_struc(ax[ii,i], pos[:,0], pos[:,1])

plt.tight_layout()
plt.show()

# 2. Bessel beams in full minimal coupling

Plane waves are the simplest exact solution of the free Maxwell equations, but they are not the most interesting one for beyond-dipole physics: their only spatial structure is the phase factor $e^{i\mathbf{k}\cdot\mathbf{r}}$ along the propagation direction. **Bessel beams** are non-diffracting solutions whose transverse profile is given by a Bessel function of the first kind $J_m$, and which carry a well-defined orbital angular momentum $m$ (twisted light) in addition to the spin angular momentum given by the helicity. This is exactly the kind of structured light for which a truncated multipolar expansion becomes questionable and for which full minimal coupling was designed.

A Bessel beam is characterized by:

* the **amplitude** $A_0$,
* the **order** $m$ (topological charge / orbital angular momentum per photon),
* the **frequency** $\omega$,
* the **helicity** ($\pm 1$, i.e. the circular polarization state),
* the **opening (cone) angle** $\theta_k$: the beam is a superposition of plane waves whose wave vectors lie on a cone of half-angle $\theta_k$ around the beam axis, so that $k_z = |\mathbf{k}|\cos\theta_k$ and $k_\perp = |\mathbf{k}|\sin\theta_k$. For $\theta_k \to 0$ the Bessel beam reduces to a plane wave.

In Octopus this is requested with the `bessel_function` keyword of the `MaxwellIncidentWaves` block, whose columns are

```
bessel_function | "field_type" | A_0 | m | omega | helicity | theta_k | "mx_envelope_name" | phase
```

(the last column, the phase, is optional). As for the plane wave, `"field_type"` must be `vector_potential` in order to be compatible with `full_minimal_coupling`, and the spatio-temporal envelope is defined in the `MaxwellFunctions` block. The position of the beam axis can be moved with the `BesselBeamAxisShift` variable.

The field we want is the following (note that `tau0` is now $1.5$ periods instead of $1$, so the pulse is a bit longer and the propagation time has to be increased accordingly):

```
omega = 270*ev
period = 0.015*fs
ampl = 0.0001
tau0 = 1.5*period # sigma
t0 = tau0*3
p_s = - t0*c
pw  = tau0*c
phase_vec = -pi/2
theta = pi*5/12

%MaxwellIncidentWaves
 bessel_function | vector_potential | -ampl*c/omega | 2.0 | omega | 1 | theta | "env_function"
%
%MaxwellFunctions
  "env_function" | mxf_gaussian_wave | 0 | omega/c | 0 | 0 | p_s | 0 |  pw
%
```

i.e. a Bessel beam of order $m = 2$, helicity $+1$, opening angle $\theta_k = 75^\circ$ and the same Gaussian envelope as before. Since we are using the analytical external source, the block has to be written with the `ExternalSource` namespace, `%ExternalSource.MaxwellIncidentWaves`, exactly as we did for the plane wave.

In [ ]:
if not os.path.exists('BESSEL_FULL_MINIMAL_DIR'):
    os.mkdir('BESSEL_FULL_MINIMAL_DIR')

In [ ]:
%%writefile BESSEL_FULL_MINIMAL_DIR/inp

CalculationMode = td
ExperimentalFeatures = yes
FromScratch = yes
RestartWallTimePeriod = 10.01
ParStates = no

%Systems
 "benzene" | electronic
%

XYZCoordinates = "benzene.xyz"
UnitsXYZFiles = angstrom_units
AllElectronType = full_delta

BoxShape = parallelepiped
benzene.Spacing = 0.1*angstrom
%benzene.Lsize
 3.5*angstrom | 3.5*angstrom | 1.5*angstrom
%

%TDOutput
 multipoles
%
%Output
 density | plane_z
%
OutputInterval = 20

omega = 270*ev
period = 0.015*fs
ampl = 0.0001
tau0 = 1.5*period # sigma
t0 = tau0*3
p_s = - t0*c
pw  = tau0*c
phase_vec = -pi/2
theta = pi*5/12

TDSystemPropagator = prop_aetrs
benzene.TDTimeStep = 0.008
TDPropagationTime = 10*period

MaxwellCouplingMode = full_minimal_coupling
AnalyticalExternalSource = yes
%ExternalSource.MaxwellIncidentWaves
 bessel_function | vector_potential | -ampl*c/omega | 2.0 | omega | 1 | theta | "env_function"
%
%MaxwellFunctions
  "env_function" | mxf_gaussian_wave | 0 | omega/c | 0 | 0 | p_s | 0 |  pw
%

In [ ]:
%%bash
mkdir -p BESSEL_FULL_MINIMAL_DIR/restart/benzene
cp -r GS_DIR/restart/* BESSEL_FULL_MINIMAL_DIR/restart/benzene
cd BESSEL_FULL_MINIMAL_DIR
cp ../GS_DIR/benzene.xyz .
export OMP_NUM_THREADS=1
mpirun -np $NPROC $OCTOPUS >& out.log
tail -n 3 out.log

Let's now compare the response of the molecule to the Bessel beam with the one obtained with the plane wave, both in full minimal coupling. Keep in mind that the two pulses have the same peak amplitude and central frequency, but different spatial structure (and a slightly longer envelope for the Bessel beam), so the comparison is qualitative.

In [ ]:
run_BESSEL_DIR = Run("BESSEL_FULL_MINIMAL_DIR") # Use Postopus to load all data produced by octopus
run_BESSEL_multipoles = run_BESSEL_DIR.benzene.td.multipoles() # Load multipole data

fig, ax = plt.subplots(figsize=(9,4.5))
run_BESSEL_multipoles.plot(x="t", y="<x>(1)", ax=ax, label="Bessel beam (m=2), full minimal")
run_PLANE_WAVE_multipoles.plot(x="t", y="<x>(1)", ax=ax, label="Plane wave, full minimal")
ax.set_ylabel('dipole moment <x> [a.u.]'); ax.set_xlabel('time [a.u.]');
ax.legend();

In [ ]:
run_BESSEL_DIR = Run("BESSEL_FULL_MINIMAL_DIR") # Use Postopus to load all data produced by octopus
run_BESSEL_multipoles = run_BESSEL_DIR.benzene.td.multipoles() # Load multipole data

fig, ax = plt.subplots(figsize=(9,4.5))
run_BESSEL_multipoles.plot(x="t", y="<x>(1)", ax=ax, label=r"$\mu_x$, Bessel beam (m=2), full minimal")
run_BESSEL_multipoles.plot(x="t", y="<y>(1)", ax=ax, label=r"$\mu_y$,  Bessel beam (m=2), full minimal")
run_BESSEL_multipoles.plot(x="t", y="<z>(1)", ax=ax, label=r"$\mu_z$, Bessel beam (m=2), full minimal")
ax.set_ylabel('dipole moment <x> [a.u.]'); ax.set_xlabel('time [a.u.]');
ax.legend();

In [ ]:
# The induced density difference for the Bessel beam
run_BESSEL_density = run_BESSEL_DIR.benzene.td.density("z=0")
diff_BESSEL_density = run_BESSEL_density - run_BESSEL_density[0]

cmap = plt.cm.terrain
vmax = 1e-6
idxs = (10, 12, 14)

fig, ax = plt.subplots(len(idxs), 2, figsize=(11, 4*len(idxs)))
for ii, idx in enumerate(idxs):
    plots = [(diff_PLANE_WAVE_density[idx], "Plane wave"),
             (diff_BESSEL_density[idx],     "Bessel beam (m=2)")]
    for i, (data, title) in enumerate(plots):
        data.plot(cmap=cmap, ax=ax[ii, i], x="x", vmax=vmax, vmin=-vmax)
        ax[ii, i].set_title(title)
        plotxy_struc(ax[ii, i], pos[:, 0], pos[:, 1])

plt.tight_layout()
plt.show()

The Bessel beam carries orbital angular momentum, and its transverse profile varies over the size of the molecule much more strongly than the one of the plane wave. Both effects are completely absent in a dipole-level description, and only partially captured by a multipolar expansion truncated at the electric quadrupole / magnetic dipole level. This is the main practical reason to use `MaxwellCouplingMode = full_minimal_coupling` when working with structured light.

### Bonus exercises:

1. Re run the calculations with the incident waves propagating along the $z$ direction (perpendicular to the plane of the molecule). What changes do you expect? What do you actually obtain?

2. Change the external field to be circularly polarized. How does the induced density change?

3. Include a calculation with `MaxwellCouplingMode = multipolar_expansion` including the electric dipole, electric quadrupole and magnetic dipole terms. Keep in mind that in this case, the external source has to be defined as an electric field (to get equivalent results, $E_0 = -A_0*\omega/c$ and the `phase_vec = 0`). How does it compare with the full minimal coupling results?

4. Now displace the molecule off-center and recompute the response of the electric quadrupole and the magnetic dipole, at the electric dipole and full minimal coupling levels. What level of theory makes the results origin-dependent?

5. Repeat the Bessel beam calculation changing the order $m$ (0, 1, 2) and the opening angle $\theta_k$. In the limit $\theta_k \rightarrow 0$, do you recover the plane wave result?

6. Displace the molecule away from the beam axis (or move the axis with `BesselBeamAxisShift`). How does the response depend on where the molecule sits with respect to the vortex core?

----

#### References

(1) Bonafé, F. P., Albar, E. I., Ohlmann, S. T., Kosheleva, V. P., Bustamante, C. M., Troisi, F., Rubio, A., & Appel, H. (2025). Full minimal coupling Maxwell-TDDFT: An ab initio framework for light-matter interaction beyond the dipole approximation. Physical Review B, 111(8), 085114. https://doi.org/10.1103/PhysRevB.111.085114

(2) Ismail-Beigi, S., Chang, E. K., & Louie, S. G. (2001). Coupling of nonlocal potentials to electromagnetic fields. Physical Review Letters, 87(8), 87402-1-87402-87404. https://doi.org/10.1103/PhysRevLett.87.087402

(3) Pickard, C. J., & Mauri, F. (2003). Nonlocal pseudopotentials and magnetic fields. Physical Review Letters, 91(19), 7-10. https://doi.org/10.1103/PhysRevLett.91.196401

(4) Jensen, S. V. B., Lund, M. M., & Madsen, L. B. (2020). Nondipole strong-field-approximation Hamiltonian. Physical Review A, 101(4), 1-14. https://doi.org/10.1103/PhysRevA.101.043408

(5) Jestädt, R., Ruggenthaler, M., Oliveira, M. J. T., Rubio, A., & Appel, H. (2019). Light-matter interactions within the Ehrenfest-Maxwell-Pauli-Kohn-Sham framework: fundamentals, implementation, and nano-optical applications. Advances in Physics, 68(4), 225-333. https://doi.org/10.1080/00018732.2019.1695875